# Download ERA5 + GOES Data for WildfireSpreadTS

Downloads hourly ERA5-Land weather and GOES-16 fire detection data for all 607 fire events, then uploads to GCS.

**Runtime**: CPU is fine (I/O bound, no GPU needed).

**Estimated time**: ~6-7h GOES, ~4-12h ERA5 (run in parallel via two notebooks, or sequentially here).

**Output in GCS**:
- `gs://BUCKET/WildfireSpreadTS_GOES/{year}/{fire_name}/{date}.npz` — (24, 3, H, W)
- `gs://BUCKET/WildfireSpreadTS_ERA5/{year}/{fire_name}/{date}.nc` — 6 vars x 24 hours

## 1. Configuration

In [ ]:
REPO_ORG = "amindell11"
REPO_NAME = "WildfireSpreadTSCreateDataset"
REPO_BRANCH = "feature/era5-goes-export"

GCS_BUCKET = "lmudl-wildfire-compilation-bucket"

# CDS API credentials (get from https://cds.climate.copernicus.eu/profile)
CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = ""  # <-- paste your personal access token here

# Which years to process (can split across multiple notebooks)
YEARS = [2018, 2019, 2020, 2021]

# Download settings
GOES_WORKERS = 6  # parallel threads for GOES (Colab has good bandwidth)
GOES_OUTPUT = "/content/goes"
ERA5_OUTPUT = "/content/era5"

## 2. Setup

In [ ]:
# Authenticate with GCS for upload
from google.colab import auth
auth.authenticate_user()

# Clone repo (private — need token)
import subprocess
result = subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], capture_output=True)

# Use Colab's credentials for GitHub
from google.colab import userdata
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except:
    GITHUB_TOKEN = input("Enter GitHub personal access token (needs repo scope): ")

!rm -rf /content/{REPO_NAME}
!git clone -b {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_ORG}/{REPO_NAME}.git /content/{REPO_NAME}
!git -C /content/{REPO_NAME} log --oneline -3

# Install dependencies
!pip install -q s3fs xarray h5netcdf netcdf4 cdsapi pyyaml tqdm

In [ ]:
# Write CDS API credentials
import os
if CDS_KEY:
    with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
        f.write(f'url: {CDS_URL}\nkey: {CDS_KEY}\n')
    print('CDS credentials written to ~/.cdsapirc')
else:
    print('WARNING: No CDS_KEY set. ERA5 download will fail. GOES will still work.')

In [ ]:
# Build config file list
config_files = [f'/content/{REPO_NAME}/config/us_fire_{y}_1e7.yml' for y in YEARS]
for f in config_files:
    assert os.path.exists(f), f'Missing: {f}'
print(f'Config files: {config_files}')

## 3. Download GOES-16 Fire Detection

Downloads from public AWS S3 (no auth needed). Uses date-first strategy with parallel threads.

Output: `(24, 3, H, W)` per day — Mask, FRP, Area at 2km, hourly composites.

In [ ]:
configs_arg = ' '.join(config_files)
!cd /content/{REPO_NAME} && pip install -q google-cloud-storage && python main_goes_download.py \
    --configs {configs_arg} \
    --gcs_bucket {GCS_BUCKET} \
    --gcs_prefix WildfireSpreadTS_GOES \
    --workers {GOES_WORKERS}

In [ ]:
# Verify GOES output in GCS
!gcloud storage ls gs://{GCS_BUCKET}/WildfireSpreadTS_GOES/ | head -10
!echo "---"
!gcloud storage ls gs://{GCS_BUCKET}/WildfireSpreadTS_GOES/**/*.npz 2>/dev/null | wc -l
!echo "total .npz files in GCS"

In [ ]:
# GOES data is already in GCS — no upload needed
print("GOES data written directly to GCS. No upload step needed.")

## 4. Download ERA5-Land Hourly Weather

Downloads from Copernicus CDS API. Requires credentials (set CDS_KEY above).

Output: NetCDF per day with 6 variables x 24 hours at ~11km resolution.

In [ ]:
!pip install -q cfgrib eccodes google-cloud-storage

# Process one year at a time (CDS queuing works better with smaller requests)
for year in YEARS:
    cfg = f'/content/{REPO_NAME}/config/us_fire_{year}_1e7.yml'
    print(f'\n=== Processing {year} ===')
    !cd /content/{REPO_NAME} && python main_era5_download.py \
        --config {cfg} \
        --gcs_bucket {GCS_BUCKET} \
        --gcs_prefix WildfireSpreadTS_ERA5

In [ ]:
# Verify ERA5 output in GCS
!gcloud storage ls gs://{GCS_BUCKET}/WildfireSpreadTS_ERA5/ | head -10
!echo "---"
!gcloud storage ls gs://{GCS_BUCKET}/WildfireSpreadTS_ERA5/**/*.nc 2>/dev/null | wc -l
!echo "total .nc files in GCS"

In [ ]:
# ERA5 data is already in GCS — no upload needed
print("ERA5 data written directly to GCS. No upload step needed.")

## 5. Summary

In [ ]:
goes_count = len(glob.glob(f'{GOES_OUTPUT}/**/*.npz', recursive=True))
era5_count = len(glob.glob(f'{ERA5_OUTPUT}/**/*.nc', recursive=True))

print(f'GOES files: {goes_count}')
print(f'ERA5 files: {era5_count}')
print(f'Expected: ~13,615 each (607 fires x ~22 days avg)')
print()
print(f'GCS locations:')
print(f'  gs://{GCS_BUCKET}/WildfireSpreadTS_GOES/')
print(f'  gs://{GCS_BUCKET}/WildfireSpreadTS_ERA5/')
print()
print('Next: use these in the training notebook alongside existing HDF5 data.')